# Satellite Data Acquisition Pipeline
**PyGeoVision v2.0 — Notebook 01**

## Real-World Problem
A research team studying Amazon deforestation needs weekly Sentinel-2 downloads
without manually navigating each provider's portal.

## Learning Objectives
1. Authenticate with 22+ satellite providers
2. Search with bbox, date range, and cloud-cover filters
3. Download in parallel with automatic post-processing
4. Inspect GeoTIFF metadata and create RGB previews
5. Use search caching for sub-second repeated queries

## Prerequisites
```bash
pip install "pygeovision[geo]" rasterio matplotlib geopandas
```

In [2]:
# !pip install ipykernel --force-reinstall
!pip install "pygeovision[geo]" rasterio matplotlib geopandas

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [1]:
# Imports
import pygeovision as pgv
import rasterio, numpy as np, matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import box
from pathlib import Path
import time, warnings
warnings.filterwarnings('ignore')

# Configuration
BASE_DIR = Path("./outputs/01_data_acquisition")
BASE_DIR.mkdir(parents=True, exist_ok=True)

STUDY_BBOX = (-52.50, -4.50, -51.50, -3.50)   # Pará state, Brazil
DATE_RANGE = ("2024-06-01", "2024-08-31")      # Dry season
CLOUD_MAX  = 10                                 # %
PROVIDERS  = ["planetary_computer"]

client = pgv.PyGeoVision()
print(client)
print(f"\nStudy area  : {STUDY_BBOX}")
print(f"Date range  : {DATE_RANGE[0]} to {DATE_RANGE[1]}")
print(f"Cloud max   : {CLOUD_MAX}%")
print(f"Output dir  : {BASE_DIR}")
pgv.__version__

PyGeoVision(v2.0.4 | pygeofetch=✓ | geoai=independent | datasets=503 | models=98 | pipelines=51 | labeling=7src | losses=10 | inference=4 | xai=4 | monitoring=3 | cloud=3·aws·azure·gcp | edge=ONNX+Jetson | vlm=CLIP+Moon | few_shot | timeseries | 3D)

Study area  : (-52.5, -4.5, -51.5, -3.5)
Date range  : 2024-06-01 to 2024-08-31
Cloud max   : 10%
Output dir  : outputs/01_data_acquisition


'2.0.4'

## Step 1: Provider Overview and Authentication

In [5]:
# # List all available providers
# all_providers  = client.data.list_providers()
# open_providers = client.data.list_providers(open_only=True)

# print(f"Total providers : {len(all_providers)}")
# print(f"Open / free     : {len(open_providers)}")
# print()
# print(f"{'Provider':<35} {'Type':>8} {'Open':>6} {'STAC':>6}")
# print("─" * 60)
# for p in all_providers[:14]:
#     name  = p.get('name', p.get('id', '?'))
#     ptype = 'SAR' if p.get('sar') else 'Optical'
#     open_ = 'Yes' if p.get('open') else 'No'
#     stac  = 'Yes' if p.get('stac') else 'No'
#     print(f"  {name:<33} {ptype:>8} {open_:>6} {stac:>6}")

In [6]:
# Add credentials (uncomment as needed)
# client.add_credentials("planetary_computer", api_key="YOUR_KEY")
# client.add_credentials("usgs", username="user", password="pass")
# client.add_credentials("copernicus", username="user", password="pass")
# client.add_credentials("planet", api_key="YOUR_PLANET_KEY")

print("Using Planetary Computer — free access, no key required")
print("Register at: https://planetarycomputer.microsoft.com/")
print()

# Test credentials
try:
    result = client.data.provider_info("planetary_computer")
    print(f"Provider: {result.get('name','Planetary Computer')}")
    print(f"Open:     {result.get('open', True)}")
    print(f"STAC:     {result.get('stac', True)}")
except Exception as e:
    print(f"Provider info: {e}")

Using Planetary Computer — free access, no key required
Register at: https://planetarycomputer.microsoft.com/

Provider: Microsoft Planetary Computer
Open:     True
STAC:     True


## Step 2: Search for Imagery

In [8]:
# Basic search
print("Searching Planetary Computer for Sentinel-2 L2A scenes...")
t0 = time.time()

results = client.search(
    bbox            = STUDY_BBOX,
    date_range      = DATE_RANGE,
    providers       = PROVIDERS,
    cloud_cover_max = CLOUD_MAX,
    collections     = ["sentinel-2-l2a"],
    sort_by         = "cloud_cover",
    limit           = 20,
    use_cache       = False,
)

elapsed = time.time() - t0
print(f"Found {len(results)} scenes in {elapsed:.2f}s\n")

if results:
    print(f"{'#':>3}  {'Date':<12} {'Cloud%':>7} {'Platform':<20} {'ID'}")
    print("─" * 72)
    for i, r in enumerate(results[:8], 1):
        print(f"  {i:>2}  {r.date:<12} {r.cloud_cover:>6.1f}%  "
              f"{str(r.satellite):<20}  {r.id[:30]}...")

2026-06-27 02:25:55,070 [INFO] pygeovision.data.fetch: Searching 1 provider(s) [planetary_computer] | bbox=(-52.5, -4.5, -51.5, -3.5) | 2024-06-01→2024-08-31 | cloud≤10%
2026-06-27 02:25:55,072 [INFO] pygeovision.data.fetch: Search complete: 20 results


Searching Planetary Computer for Sentinel-2 L2A scenes...
Found 20 scenes in 0.00s

  #  Date          Cloud% Platform             ID
────────────────────────────────────────────────────────────────────────
   1  2024-06-21     10.7%  Sentinel-2B           S2B_MSIL2A_20240621T134709_R02...
   2  2024-08-25     10.6%  Sentinel-2A           S2A_MSIL2A_20240825T134701_R02...
   3  2024-06-21     10.5%  Sentinel-2B           S2B_MSIL2A_20240621T134709_R02...
   4  2024-07-21     10.4%  Sentinel-2B           S2B_MSIL2A_20240721T134709_R02...
   5  2024-08-25      6.3%  Sentinel-2A           S2A_MSIL2A_20240825T134701_R02...
   6  2024-08-30      5.9%  Sentinel-2B           S2B_MSIL2A_20240830T134709_R02...
   7  2024-06-21      4.3%  Sentinel-2B           S2B_MSIL2A_20240621T134709_R02...
   8  2024-06-21      3.5%  Sentinel-2B           S2B_MSIL2A_20240621T134709_R02...


In [11]:
# Inspect best scene
if results:
    best = results[0]
    print("Best scene details:")
    print(f"  ID          : {best.id}")
    print(f"  Date        : {best.date}")
    print(f"  Cloud cover : {best.cloud_cover:.1f}%")
    print(f"  Platform    : {best.satellite}")
    print(f"  Resolution  : {best.resolution_m}m")
    # print(f"  CRS         : {best.crs}")
    # print(f"  Bands       : {best.bands}")
    print(f"  Provider    : {best.provider}")
    print(f"  Collection  : {best.collection}")
else:
    print("No results — using demo data")

Best scene details:
  ID          : S2B_MSIL2A_20240621T134709_R024_T22MCB_20240621T182529
  Date        : 2024-06-21
  Cloud cover : 10.7%
  Platform    : Sentinel-2B
  Resolution  : 10.0m
  Provider    : planetary_computer
  Collection  : sentinel-2-l2a


In [13]:
# Advanced search with CQL2 filter
advanced = client.search(
    bbox            = STUDY_BBOX,
    date_range      = DATE_RANGE,
    providers       = PROVIDERS,
    cloud_cover_max = 15,
    cql2_filter     = "eo:cloud_cover < 15 AND platform = 'sentinel-2b'",
    sort_by         = "cloud_cover",
    limit           =2,
)
print(f"CQL2-filtered results (Sentinel-2B only): {len(advanced)} scenes")
for r in advanced[:3]:
    print(f"  {r.date}  cloud={r.cloud_cover:.1f}%  {r.satellite}  {r.id[:30]}...")

CQL2-filtered results (Sentinel-2B only): 2 scenes
  2024-08-30  cloud=12.5%  Sentinel-2B  S2B_MSIL2A_20240830T134709_R02...
  2024-08-30  cloud=5.9%  Sentinel-2B  S2B_MSIL2A_20240830T134709_R02...


## Step 3: Download with Post-Processing

In [ ]:
SCENE_DIR = BASE_DIR / "scenes"
SCENE_DIR.mkdir(exist_ok=True)

if results:
    print(f"Downloading {min(2, len(results))} best scenes...")
    downloads = client.download(
        results[:2],
        output_dir   = SCENE_DIR,
        post_process = [
            "reproject:EPSG:32722",   # UTM Zone 22S — covers Pará state
            "cog",                    # Cloud-Optimized GeoTIFF
        ],
        parallel     = 2,
        overwrite    = False,
    )
    print()
    for d in downloads:
        status = "OK" if d.success else "FAIL"
        path   = Path(d.path) if d.path else None
        size   = f"{path.stat().st_size/1e6:.1f}MB" if (path and path.exists()) else "N/A"
        print(f"  [{status}]  {d.scene_id[:32]}  {size}")
else:
    downloads = []
    print("No results — skipping download")

In [2]:
import pygeovision 
pygeovision.__version__

'2.0.3'

In [ ]:
SCENE_DIR = Path("/home/mrtenkorang/pygeovision_v2_complete/projects/outputs/01_data_acquisition/scenes/planetary_computer")
OUTPUT_DIR = Path("/home/mrtenkorang/pygeovision_v2_complete/projects/outputs/01_data_acquisition/ready/")
result = client.preprocess.pipeline(
        input_path  = SCENE_DIR,
        output_path = "ready.tif",
        stack_bands = ["B02","B03","B04","B08","B11","B12"],
        bbox        = STUDY_BBOX,
        
        # scl_path    = "./downloads/S2C_20240628/SCL.tif",
        normalise   = "scale_factor",
    )

2026-06-27 03:23:54,220 [INFO] pygeovision.preprocess.core: Stacked 6 bands → /tmp/pgv_preprocess_e3hfebq9/step_00_stacked.tif
2026-06-27 03:24:02,422 [INFO] pygeovision.preprocess.core: Clipped to bbox: 10980x10980 → 7623x2293 (14.5% of original) → /tmp/pgv_preprocess_e3hfebq9/step_01_clipped.tif
2026-06-27 03:24:13,571 [INFO] pygeovision.preprocess.core: Normalised (scale_factor) → /tmp/pgv_preprocess_e3hfebq9/step_02_norm.tif
2026-06-27 03:24:13,709 [INFO] pygeovision.preprocess.core: Pipeline complete: /home/mrtenkorang/pygeovision_v2_complete/projects/outputs/01_data_acquisition/ready | shape=(6, 2293, 7623) | res=10.0m | steps=['stack(6 bands)', 'clip_bbox(-52.5, -4.5, -51.5, -3.5)', 'normalise(scale_factor)']


In [12]:
from pathlib import Path
import os
import json
from pathlib import Path
from pygeovision import PyGeoVision
from pygeovision.preprocess import Preprocessor
from pygeovision.data.postprocess import PostProcessor
import rasterio
from shapely.geometry import box
import geopandas as gpd

# Initialize clients
client = PyGeoVision()
pre = Preprocessor()
post = PostProcessor()

# Set up paths
SCENE_DIR = Path("/home/mrtenkorang/pygeovision_v2_complete/projects/outputs/01_data_acquisition/scenes/planetary_computer")
OUTPUT_DIR = Path("/home/mrtenkorang/pygeovision_v2_complete/projects/outputs/01_data_acquisition/ready/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================================
# 1. FIND YOUR ACTUAL DATA EXTENT FIRST
# ============================================================================

print("=" * 60)
print("CHECKING DATA EXTENT")
print("=" * 60)



# ============================================================================
# 2. FULL PREPROCESSING PIPELINE (Fixed)
# ============================================================================

print("\n" + "=" * 60)
print("STARTING PREPROCESSING PIPELINE")
print("=" * 60)

# Find the SCL file
scl_files = list(SCENE_DIR.rglob("*SCL*.tif"))
scl_path = str(scl_files[0]) if scl_files else None
if scl_path:
    print(f"✅ Found SCL file: {scl_path}")
else:
    print("⚠️  No SCL file found - skipping cloud masking")

# Make sure output directory exists
output_path = OUTPUT_DIR / "ready_6band.tif"
print(f"📁 Output will be saved to: {output_path}")

# Run the main preprocessing pipeline
try:
    result = client.preprocess.pipeline(
        input_path=str(SCENE_DIR),
        output_path=str(output_path),
        stack_bands=["B02", "B03", "B04", "B08", "B11", "B12"],
        bbox=STUDY_BBOX,
        bbox_crs="EPSG:4326",  # Specify that bbox is in WGS84
        scl_path=scl_path,  # Use SCL for cloud masking
        scl_keep_classes=[4, 5, 6],  # Keep vegetation, bare soil, water
        normalise="scale_factor",
        scale_factor=10000.0,  # Sentinel-2 L2A DN to reflectance
        # resample_m=10.0,  # Uncomment if you want to resample to 10m
        keep_intermediates=False,
    )

    print(f"\n✅ Preprocessing complete!")
    print(f"   Output: {result['output_path']}")
    print(f"   Shape: {result['shape']}")
    print(f"   Resolution: {result['resolution_m']:.1f}m")
    print(f"   Steps applied: {result['steps_applied']}")

except ValueError as e:
    if "shapes outside bounds of raster" in str(e):
        print("\n❌ Bounding box does not overlap with raster!")
        print(f"   Your bbox: {STUDY_BBOX}")
        
        # Try to automatically find a valid bbox
        print("\n🔄 Attempting to use full raster extent...")
        result = client.preprocess.pipeline(
            input_path=str(SCENE_DIR),
            output_path=str(output_path),
            stack_bands=["B02", "B03", "B04", "B08", "B11", "B12"],
            # Don't clip - use full extent
            scl_path=scl_path,
            scl_keep_classes=[4, 5, 6],
            normalise="scale_factor",
            scale_factor=10000.0,
            keep_intermediates=False,
        )
        print(f"\n✅ Preprocessing complete (full extent)!")
        print(f"   Output: {result['output_path']}")
        print(f"   Shape: {result['shape']}")
    else:
        raise

# ============================================================================
# 3. VERIFY OUTPUT
# ============================================================================

print("\n" + "=" * 60)
print("VERIFYING OUTPUT")
print("=" * 60)

if output_path.exists():
    file_size = output_path.stat().st_size / (1024 * 1024)
    print(f"✅ Output file exists: {output_path.name} ({file_size:.2f} MB)")
    
    # Get info about the output
    info = pre.info(str(output_path))
    print(f"\n📊 Output Info:")
    print(f"   CRS: {info['crs']}")
    print(f"   Shape: {info['shape']}")
    print(f"   Resolution: {info['resolution_m']:.2f}m")
    print(f"   Bands: {info['bands']}")
    print(f"   Value range: {info['value_range']}")
else:
    print("❌ Output file was not created!")

# ============================================================================
# 2. ADDITIONAL PREPROCESSING STEPS (Optional)
# ============================================================================

print("\n" + "=" * 60)
print("ADDITIONAL PREPROCESSING OPTIONS")
print("=" * 60)

# # 2a. Clip to polygon (if you have a GeoJSON)
# geojson_path = Path("/path/to/study_area.geojson")
# if geojson_path.exists():
#     polygon_clipped = pre.clip_to_polygon(
#         str(OUTPUT_DIR / "ready_6band.tif"),
#         str(geojson_path),
#         output_path=str(OUTPUT_DIR / "ready_polygon_clipped.tif"),
#     )
#     print(f"✅ Polygon clip: {polygon_clipped}")

# # 2b. Resample to specific resolution
# resampled = pre.resample(
#     str(OUTPUT_DIR / "ready_6band.tif"),
#     resolution_m=10.0,
#     output_path=str(OUTPUT_DIR / "ready_10m.tif"),
#     resampling="bilinear",
# )
# print(f"✅ Resampled: {resampled}")

# # 2c. Create a COG version
# cog_path = post.to_cog(
#     str(OUTPUT_DIR / "ready_6band.tif"),
#     output_path=str(OUTPUT_DIR / "ready_cog.tif"),
# )
# print(f"✅ COG created: {cog_path}")

# # 2d. Get raster info
# info = pre.info(str(OUTPUT_DIR / "ready_6band.tif"))
# print(f"✅ Raster info:")
# for key, value in info.items():
#     print(f"   {key}: {value}")

# ============================================================================
# 3. POSTPROCESSING PIPELINE (for model predictions)
# ============================================================================

print("\n" + "=" * 60)
print("POSTPROCESSING PIPELINE")
print("=" * 60)

# For this example, we'll create a sample prediction raster
# In practice, this would be your model output

# 3a. Vectorise a prediction raster (if you have one)
# prediction_path = OUTPUT_DIR / "prediction.tif"  # Replace with actual prediction
# if prediction_path.exists():
#     print("\n3a. Vectorising prediction...")
#     vectorized = post.vectorise(
#         str(prediction_path),
#         str(OUTPUT_DIR / "predictions.geojson"),
#         target_class=1,  # Class for buildings
#         min_area_m2=25.0,
#         simplify_tolerance=0.5,
#     )
#     print(f"✅ Vectorised: {vectorized}")

#     # 3b. Smooth polygons
#     print("\n3b. Smoothing polygons...")
#     smoothed = post.smooth(
#         str(OUTPUT_DIR / "predictions.geojson"),
#         str(OUTPUT_DIR / "predictions_smooth.geojson"),
#         tolerance=0.5,
#     )
#     print(f"✅ Smoothed: {smoothed}")

#     # 3c. Regularise buildings
#     print("\n3c. Regularising buildings...")
#     regularised = post.regularise_buildings(
#         str(OUTPUT_DIR / "predictions_smooth.geojson"),
#         str(OUTPUT_DIR / "buildings_regularised.geojson"),
#         min_area_m2=10.0,
#     )
#     print(f"✅ Regularised: {regularised}")

#     # 3d. Dissolve adjacent polygons
#     print("\n3d. Dissolving polygons...")
#     dissolved = post.dissolve(
#         str(OUTPUT_DIR / "buildings_regularised.geojson"),
#         str(OUTPUT_DIR / "buildings_dissolved.geojson"),
#         by_field="class",
#     )
#     print(f"✅ Dissolved: {dissolved}")

#     # 3e. Export to different formats
#     print("\n3e. Exporting to formats...")
#     for fmt in ["shp", "gpkg", "kml"]:
#         try:
#             exported = post.export(
#                 str(OUTPUT_DIR / "buildings_regularised.geojson"),
#                 str(OUTPUT_DIR / f"buildings.{fmt}"),
#                 fmt=fmt,
#             )
#             print(f"   ✅ Exported to {fmt.upper()}: {exported}")
#         except Exception as e:
#             print(f"   ⚠️  Export to {fmt.upper()} failed: {e}")

# # 3f. Class statistics (if you have a classification raster)
# if prediction_path.exists():
#     print("\n3f. Computing class statistics...")
#     stats = post.class_statistics(str(prediction_path))
#     print("📊 Class Statistics:")
#     for class_id, info in stats.items():
#         print(f"   Class {class_id}:")
#         print(f"      Pixels: {info['pixels']:,}")
#         print(f"      Area: {info['area_ha']:.1f} ha ({info['area_km2']:.2f} km²)")
#         print(f"      Coverage: {info['pct']:.1f}%")

# # ============================================================================
# 4. ZONAL STATISTICS
# ============================================================================

print("\n" + "=" * 60)
print("ZONAL STATISTICS")
print("=" * 60)

# If you have a vector layer with zones
# zones_path = Path("/path/to/zones.geojson")
# raster_path = OUTPUT_DIR / "ready_6band.tif"

# if zones_path.exists() and raster_path.exists():
#     print("Computing zonal statistics...")
#     zonal_results = post.zonal_statistics(
#         str(raster_path),
#         str(zones_path),
#         output_path=str(OUTPUT_DIR / "zonal_stats.geojson"),
#         stats=["mean", "std", "min", "max", "count"],
#         band=1,  # Band 1 (Blue)
#     )
#     print(f"✅ Zonal statistics: {zonal_results}")

# ============================================================================
# 5. ACCURACY ASSESSMENT
# ============================================================================

print("\n" + "=" * 60)
print("ACCURACY ASSESSMENT")
print("=" * 60)

# If you have a reference raster
# reference_path = Path("/path/to/reference.tif")
# if prediction_path.exists() and reference_path.exists():
#     print("Computing accuracy metrics...")
#     acc_report = post.accuracy_assessment(
#         str(prediction_path),
#         str(reference_path),
#         class_names=["Background", "Building", "Road", "Water", "Vegetation"],
#     )
    
#     print(f"\n📊 Accuracy Report:")
#     print(f"   Overall Accuracy: {acc_report['overall_accuracy']:.4f}")
#     print(f"   Cohen's Kappa: {acc_report['kappa']:.4f}")
#     print(f"   Mean IoU: {acc_report['mean_iou']:.4f}")
    
#     print("\n   Per-class metrics:")
#     for class_id, metrics in acc_report['per_class'].items():
#         name = acc_report['class_names'][class_id] if class_id < len(acc_report['class_names']) else str(class_id)
#         print(f"   {name}:")
#         print(f"      Precision: {metrics['precision']:.4f}")
#         print(f"      Recall: {metrics['recall']:.4f}")
#         print(f"      F1-Score: {metrics['f1']:.4f}")
#         print(f"      IoU: {metrics['iou']:.4f}")

# ============================================================================
# 6. GENERATE REPORT
# ============================================================================

# print("\n" + "=" * 60)
# print("GENERATING REPORT")
# print("=" * 60)

# if prediction_path.exists():
#     report_path = post.generate_report(
#         str(prediction_path),
#         str(OUTPUT_DIR / "report.html"),
#         reference_path=str(reference_path) if reference_path.exists() else None,
#         class_names=["Background", "Building", "Road", "Water", "Vegetation"],
#         fmt="html",
#     )
#     print(f"✅ HTML Report: {report_path}")
    
#     # Also generate JSON report
#     json_report = post.generate_report(
#         str(prediction_path),
#         str(OUTPUT_DIR / "report.json"),
#         fmt="json",
#     )
#     print(f"✅ JSON Report: {json_report}")

# ============================================================================
# 7. OUTPUT SUMMARY
# ============================================================================

print("\n" + "=" * 60)
print("✅ PIPELINE COMPLETE")
print("=" * 60)

# List all outputs
print("\n📁 Generated files:")
for file in sorted(OUTPUT_DIR.glob("*")):
    if file.is_file():
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"   {file.name} ({size_mb:.2f} MB)")

# Save pipeline summary
summary = {
    "preprocessing": result,
    "output_dir": str(OUTPUT_DIR),
    "files_generated": [str(f.name) for f in OUTPUT_DIR.glob("*") if f.is_file()],
}

summary_path = OUTPUT_DIR / "pipeline_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"\n📄 Pipeline summary saved to: {summary_path}")

print("\n🎉 All preprocessing and postprocessing steps completed successfully!")

CHECKING DATA EXTENT

STARTING PREPROCESSING PIPELINE
✅ Found SCL file: /home/mrtenkorang/pygeovision_v2_complete/projects/outputs/01_data_acquisition/scenes/planetary_computer/T22MCB_20240621T134709_SCL_20m_EPSG_32722.tif
📁 Output will be saved to: /home/mrtenkorang/pygeovision_v2_complete/projects/outputs/01_data_acquisition/ready/ready_6band.tif


2026-06-27 03:40:02,025 [INFO] pygeovision.preprocess.core: Stacked 6 bands → /tmp/pgv_preprocess_kstlr3bc/step_00_stacked.tif


ValueError: Input shapes do not overlap raster.

In [ ]:
# Post-processing reference
print("Post-processing steps:")
steps = [
    ("reproject:EPSG:XXXX", "Reproject to any CRS"),
    ("cog",                 "Convert to Cloud-Optimized GeoTIFF"),
    ("cloud_mask",          "Apply scene classification layer"),
    ("clip",                "Clip to search bbox"),
    ("normalise",           "Scale values to [0,1] float32"),
    ("stack",               "Stack all bands into one file"),
]
for step, desc in steps:
    print(f"  {step:<30}  {desc}")

print()
print("Example with multiple steps:")
print('  post_process=["reproject:EPSG:32618","cloud_mask","cog"]')

## Step 4: Inspect and Visualise Downloaded Scene

In [ ]:
scene_files = sorted(SCENE_DIR.glob("*.tif"))
print(f"Downloaded files: {len(scene_files)}")

if scene_files:
    scene_path = scene_files[0]
    with rasterio.open(scene_path) as src:
        meta = src.meta
        print(f"\nFile      : {scene_path.name}")
        print(f"Bands     : {src.count}")
        print(f"Shape     : {src.height} x {src.width} px")
        print(f"CRS       : {src.crs}")
        print(f"Resolution: {src.res[0]:.1f} x {src.res[1]:.1f} m")
        print(f"Dtype     : {src.dtypes[0]}")
        n_bands = src.count
        rgb_idx = [min(3,n_bands), min(2,n_bands), min(1,n_bands)]
        data = src.read(rgb_idx).astype(np.float32)
        if n_bands >= 4:
            nir = src.read(4).astype(np.float32)
        else:
            nir = None
else:
    # Synthetic demo
    scene_path = BASE_DIR / "demo_scene.tif"
    H, W = 512, 512
    np.random.seed(42)
    data = np.stack([
        np.random.randint(300, 2500, (H,W), dtype=np.uint16).astype(np.float32),
        np.random.randint(400, 3000, (H,W), dtype=np.uint16).astype(np.float32),
        np.random.randint(500, 2000, (H,W), dtype=np.uint16).astype(np.float32),
    ])
    nir = np.random.randint(3000, 7000, (H,W), dtype=np.uint16).astype(np.float32)
    import rasterio.transform as rt
    with rasterio.open(scene_path, 'w', driver='GTiff', height=H, width=W,
                       count=3, dtype='uint16', crs='EPSG:32722',
                       transform=rt.from_bounds(-52.5,-4.5,-51.5,-3.5,W,H)) as dst:
        dst.write(data.astype(np.uint16))
    print(f"Created synthetic demo scene: {scene_path}")

In [ ]:
# Visualise RGB, false colour, and NDVI
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

def stretch(arr, pct=(2,98)):
    flat = arr[arr > 0].ravel() if (arr > 0).any() else arr.ravel()
    lo, hi = np.percentile(flat, pct) if len(flat) else (0, 1)
    return np.clip((arr - lo) / (hi - lo + 1e-8), 0, 1)

# True colour
rgb = data.transpose(1, 2, 0)
axes[0].imshow(stretch(rgb))
axes[0].set_title("True Colour (R-G-B)", fontsize=12, fontweight='bold')
axes[0].axis('off')

# False colour (NIR-R-G)
if nir is not None:
    fc = np.stack([nir, data[0], data[1]]).transpose(1, 2, 0)
    axes[1].imshow(stretch(fc))
    axes[1].set_title("False Colour (NIR-R-G)", fontsize=12, fontweight='bold')
else:
    axes[1].imshow(data[0], cmap='viridis')
    axes[1].set_title("Band 1", fontsize=12)
axes[1].axis('off')

# NDVI
if nir is not None:
    red  = data[0].astype(np.float32)
    ndvi = np.where((nir + red) > 0, (nir - red) / (nir + red + 1e-10), 0)
    ndvi = np.clip(ndvi, -1, 1)
else:
    # Synthetic NDVI for demo
    ndvi = np.random.uniform(-0.2, 0.9, data.shape[1:])

im = axes[2].imshow(ndvi, cmap='RdYlGn', vmin=-0.5, vmax=1.0)
plt.colorbar(im, ax=axes[2], label='NDVI', fraction=0.046, pad=0.04)
mean_ndvi = float(ndvi[(ndvi > -1) & (ndvi < 1)].mean())
axes[2].set_title(f"NDVI  (mean = {mean_ndvi:.3f})", fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.suptitle(f"Sentinel-2 Preview — Amazon Study Area", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE_DIR / "scene_preview.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {BASE_DIR/'scene_preview.png'}")

## Step 5: Study Area Map

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

# Study bbox
bbox_gdf = gpd.GeoDataFrame({'geometry': [box(*STUDY_BBOX)]}, crs='EPSG:4326')
bbox_gdf.boundary.plot(ax=ax, color='red', linewidth=3, label='Study area')

# Scene footprints
if results:
    fps = [{'geometry': box(*r.bbox), 'cloud': r.cloud_cover}
           for r in results[:10] if r.bbox]
    if fps:
        gdf_s = gpd.GeoDataFrame(fps, crs='EPSG:4326')
        gdf_s.plot(ax=ax, column='cloud', cmap='Blues', alpha=0.35,
                   edgecolor='steelblue', linewidth=0.7,
                   legend=True, legend_kwds={'label':'Cloud Cover (%)',
                                             'shrink':0.6})

ax.set_xlim(STUDY_BBOX[0]-0.5, STUDY_BBOX[2]+0.5)
ax.set_ylim(STUDY_BBOX[1]-0.5, STUDY_BBOX[3]+0.5)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(
    f'Study Area — Pará State, Amazon Basin\n'
    f'{len(results)} scenes found  |  Cloud ≤ {CLOUD_MAX}%  |  {DATE_RANGE[0]} – {DATE_RANGE[1]}',
    fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(BASE_DIR / "study_area_map.png", dpi=150, bbox_inches='tight')
plt.show()

## Step 6: Cache and Performance

In [ ]:
print("Search cache performance:\n")

t_start = time.time()
_r1 = client.search(bbox=STUDY_BBOX, date_range=DATE_RANGE,
                     providers=PROVIDERS, cloud_cover_max=CLOUD_MAX, use_cache=False)
t_fresh = time.time() - t_start

t_start = time.time()
_r2 = client.search(bbox=STUDY_BBOX, date_range=DATE_RANGE,
                     providers=PROVIDERS, cloud_cover_max=CLOUD_MAX, use_cache=True)
t_cache = time.time() - t_start

print(f"  Fresh search  : {t_fresh:.3f}s")
print(f"  Cached search : {t_cache:.3f}s")
speedup = t_fresh / max(t_cache, 0.001)
print(f"  Speedup       : {speedup:.0f}x")

stats = client.cache_stats()
print(f"\nCache statistics:")
for k, v in stats.items():
    print(f"  {k}: {v}")

## Summary

In [ ]:
print("=" * 60)
print("NOTEBOOK 01 COMPLETE — Satellite Data Acquisition")
print("=" * 60)
print()
print(f"Scenes found     : {len(results)}")
print(f"Scenes downloaded: {len([d for d in downloads if d.success])}")
print(f"Output location  : {BASE_DIR}")
print()
print("Next steps:")
print("  02_building_footprint_extraction.ipynb")
print("  03_land_cover_classification_prithvi.ipynb")
print("  04_change_detection_temporal_analysis.ipynb")